In [1]:
import polars as pl
import csv
from procompa import get_project_root

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

get seqeunces of all proteins in complex portal or yeastmap and cdjust fromat so i canrun mmseq on them

In [3]:
input_csv = data_dir / "iPTM_and_pLDDT/all_CF_YM_yeast_proteins_uniprot_mapped_sequences.csv"
output_fasta = data_dir / "iPTM_and_pLDDT/all_CF_YM_yeast_proteins.fasta"

with open(input_csv) as f_in, open(output_fasta, "w") as f_out:
    reader = csv.DictReader(f_in)
    for row in reader:
        uniprot_id = row["uniprot_id"].strip()
        sequence = row["sequence"].strip()
        f_out.write(f">{uniprot_id}\n{sequence}\n")

prepare YM complex lift for pdb breakdown (how much of that complex is structurally supported)

In [3]:
YM_complexes = pl.read_csv(data_dir/ "Complex_Portal/YeastMap/yeast.MAP_complexes_wConfidenceScores_wGenenames_total779_20251214.csv")

In [4]:
YM_complexes.head(3)

yeastMAP_ID,ComplexConfidence,UniProt_ACCs,Genenames
str,i64,str,str
"""S.cere_0000""",1,"""P25359 P38792 Q08162 P48240 Q0…","""RRP43 RRP4 DIS3 MTR3 RRP40 SKI…"
"""S.cere_0001""",1,"""P24783 P47019 P10664 Q02326 P4…","""DBP2 ALB1 RPL4A RPL6A RPL10 RP…"
"""S.cere_0002""",1,"""P32585 Q06213 P33308 P53114 P3…","""SRB5 NUT2 CSE2 NUT1 SRB2 MED7 …"


In [6]:
yeastmap_complexes_for_script4 = (
    YM_complexes.select(
        pl.concat_str(
            [
                pl.lit("pred_CPX_"),
                (pl.col("yeastMAP_ID").str.extract(r"(\d+)$").cast(pl.Int64) + 1).cast(pl.Utf8),
            ]
        ).alias("#Complex ac"),
        pl.col("UniProt_ACCs").alias("Cleaned Identifiers"),
    )
    .with_columns(
        pl.col("Cleaned Identifiers")
        .str.split(" ")
        .list.eval(pl.element().filter(pl.element() != "").str.to_lowercase())
        .list.n_unique()
        .alias("n_proteins")
    )
)

assert yeastmap_complexes_for_script4["#Complex ac"].is_unique().all(), "duplicate complex ids"
assert (yeastmap_complexes_for_script4["n_proteins"] >= 1).all(), "complex without proteins"

yeastmap_complexes_for_script4.write_csv(data_dir/ "Complex_Portal/YeastMap/YM_complexes_cleaned.csv")